# 模型对话演示

本 Notebook 演示如何使用智谱 SDK 进行基础对话：

- 单轮对话
- 多轮对话
- 流式输出

所有配置从 `config.json` 读取。

## 0. 读取配置文件

In [1]:
import json
import os

CONFIG_FILE = "config.json"

config = {
    "zhipu_api_key": "your-zhipu-api-key-here",
    "chat_model": "glm-4-flash"
}

if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        config.update(json.load(f))

ZHIPU_API_KEY = config["zhipu_api_key"]
CHAT_MODEL = config["chat_model"]

print(f"智谱 API Key 是否已设置: {bool(ZHIPU_API_KEY and not ZHIPU_API_KEY.startswith('your-'))}")
print(f"使用模型: {CHAT_MODEL}")

智谱 API Key 是否已设置: True
使用模型: glm-4.6v-flashx


## 1. 初始化智谱客户端

In [2]:
from zhipuai import ZhipuAI

client = ZhipuAI(api_key=ZHIPU_API_KEY)
print("智谱客户端初始化完成")

智谱客户端初始化完成


## 2. 单轮对话

In [3]:
def single_chat(message: str, model: str = CHAT_MODEL) -> str:
    """单次问答，返回模型完整回复。"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": message}],
        temperature=0.7,
        max_tokens=1024,
    )
    return response.choices[0].message.content


question = "请用一句话解释什么是大语言模型。"
answer = single_chat(question)
print(f"问题: {question}")
print(f"回答: {answer}")

问题: 请用一句话解释什么是大语言模型。
回答: 
大语言模型是通过对海量文本数据进行深度学习预训练的模型，具备理解和生成人类自然语言的能力。


## 3. 带 System Prompt 的对话

In [4]:
def chat_with_system(user_message: str, system_prompt: str, model: str = CHAT_MODEL) -> str:
    """带角色设定的单次问答。"""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        temperature=0.7,
        max_tokens=1024,
    )
    return response.choices[0].message.content


system_prompt = "你是一位耐心的 Python 编程老师，回答要简洁，并给出代码示例。"
user_message = "Python 里的列表推导式是什么？"
answer = chat_with_system(user_message, system_prompt)
print(f"System Prompt: {system_prompt}")
print(f"问题: {user_message}")
print(f"回答:\n{answer}")

System Prompt: 你是一位耐心的 Python 编程老师，回答要简洁，并给出代码示例。
问题: Python 里的列表推导式是什么？
回答:

Python 里的**列表推导式**是一种**简洁的语法**，用于从一个可迭代对象（如列表、元组、字符串等）快速创建新列表。它比传统 `for` 循环更紧凑、可读性更好。  

### 基本语法  
```python
[表达式 for 变量 in 可迭代对象]
```  
- `表达式`：对每个元素的计算逻辑（如 `x * x`）。  
- `变量`：遍历可迭代对象的循环变量。  
- `可迭代对象`：如 `range()`、列表、字符串等。  


### 示例  
1. **生成 1~10 的平方数**  
   ```python
   squares = [x * x for x in range(1, 11)]
   print(squares)  # 输出: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
   ```  

2. **筛选偶数**  
   ```python
   evens = [x for x in range(1, 11) if x % 2 == 0]
   print(evens)  # 输出: [2, 4, 6, 8, 10]
   ```  

3. **生成两个列表的笛卡尔积**  
   ```python
   list1 = [1, 2]
   list2 = ['a', 'b']
   pairs = [(x, y) for x in list1 for y in list2]
   print(pairs)  # 输出: [(1, 'a'), (1, '


## 4. 多轮对话

In [6]:
class SimpleChatSession:
    """维护多轮对话历史。"""
    def __init__(self, model: str = CHAT_MODEL, system_prompt: str = None):
        self.model = model
        self.messages = []
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def send(self, user_message: str) -> str:
        self.messages.append({"role": "user", "content": user_message})
        response = client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.7,
            max_tokens=1024,
        )
        reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        return reply


session = SimpleChatSession(system_prompt="你是公司的智能助理，回答简短。")

print("Round 1:")
print(session.send("我们公司年假怎么算？"))

print("\nRound 2:")
print(session.send("那入职第 3 年呢？"))

print("\n当前对话历史:")
for m in session.messages:
    print(f"{m['role']}: {m['content'][:50]}...")

Round 1:

年假一般按工龄计算（如每满1年工龄对应1天年假，具体天数以公司制度为准），计算方式通常从入职之日起算。

Round 2:


当前对话历史:
system: 你是公司的智能助理，回答简短。...
user: 我们公司年假怎么算？...
assistant: 
年假一般按工龄计算（如每满1年工龄对应1天年假，具体天数以公司制度为准），计算方式通常从入职之日起...
user: 那入职第 3 年呢？...
assistant: ...


## 5. 流式输出

In [7]:
def stream_chat(message: str, model: str = CHAT_MODEL):
    """流式输出，逐字打印模型回复。"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": message}],
        temperature=0.7,
        max_tokens=1024,
        stream=True,
    )
    print("模型回复: ", end="", flush=True)
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
    print()


stream_chat("请用三句话介绍人工智能。")

模型回复: 
人工智能是模拟、延伸和扩展人类智能的先进技术，通过算法与大数据实现自主决策和学习能力；它在医疗诊断、自动驾驶、智能制造等领域广泛应用，显著提升效率并创造新价值；随着技术迭代，人工智能正深刻重塑社会生产生活方式，同时需关注伦理与安全等挑战的应对。


## 6. 对比不同 temperature

In [10]:
def chat_with_temperature(message: str, temperature: float, model: str = CHAT_MODEL) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": message}],
        temperature=temperature,
        max_tokens=512,
    )
    return response.choices[0].message.content


message = "用一句话描述春天。"
for temp in [0.0, 0.7, 1.2]:
    print(f"\ntemperature={temp}:")
    answer = chat_with_temperature(message, temp)
    print(answer if answer.strip() else "（模型返回空内容）")


temperature=0.0:

春天是冰雪消融、草木萌发、万物复苏，充满希望与生机的美好时节。

temperature=0.7:

春天是冰雪消融、草木萌发、百花齐放，充满生机与希望的时节。

temperature=1.2:

春天是万物复苏、百花争艳、充满生机的季节。
